## Unstructured Data Cleaning

In this notebook we will apply different generic transformations on images, the only unstructured data we have. Specifically, we will remove corrupted images and normalize their size, while intentionally avoiding resolution changes to prevent information loss. All these transformations will be performed using Apache Spark.

**Importing Useful Libraries**

In [1]:
import os
import io
import boto3
from PIL import Image
from dotenv import load_dotenv
from pyspark.sql import SparkSession

# Load .env file
load_dotenv()

# Read environment variables
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

In [2]:
# Setup S3 client for MinIO (MinIO implements Amazon S3 API)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint, # MinIO API endpoint
    aws_access_key_id=access_key, # User name
    aws_secret_access_key=secret_key, # Password
)

**Copying Images Into Trusted Zone**

In [3]:
# This function copies all objects from a bucket (under a prefix) into another bucket, creating the destination bucket if it doesn’t exist.
def replicate_bucket(src_bucket, dest_bucket, src_prefix=""):

    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=src_bucket, Prefix=src_prefix):

        for obj in page.get("Contents", []):

            key = obj["Key"]

            # Keep only from "image/" onward
            if "image/" in key:
                new_key = key.split("image/", 1)[1]
                new_key = f"image/{new_key}"
            else:
                continue  # skip anything unexpected

            copy_source = {
                "Bucket": src_bucket,
                "Key": key
            }

            s3.copy_object(
                Bucket=dest_bucket,
                Key=key,
                CopySource=copy_source
            )

            print(f"Copied: {key}")

In [4]:
# Replicate images from Landing Zone to Trusted Zone
replicate_bucket(src_bucket = "landing-zone", src_prefix="persistent-landing/unstructured/image", dest_bucket = "trusted-zone")

Copied: persistent-landing/unstructured/image/
Copied: persistent-landing/unstructured/image/image_1777660672417.jpg
Copied: persistent-landing/unstructured/image/image_1777660672462.jpg
Copied: persistent-landing/unstructured/image/image_1777660672511.jpg
Copied: persistent-landing/unstructured/image/image_1777660672555.jpg
Copied: persistent-landing/unstructured/image/image_1777660672596.jpg
Copied: persistent-landing/unstructured/image/image_1777660672642.jpg
Copied: persistent-landing/unstructured/image/image_1777660672690.jpg
Copied: persistent-landing/unstructured/image/image_1777660672770.jpg
Copied: persistent-landing/unstructured/image/image_1777660672814.jpg
Copied: persistent-landing/unstructured/image/image_1777660672861.jpg
Copied: persistent-landing/unstructured/image/image_1777660672906.jpg
Copied: persistent-landing/unstructured/image/image_1777660672970.jpg
Copied: persistent-landing/unstructured/image/image_1777660673018.jpg
Copied: persistent-landing/unstructured/ima

In [5]:
# Check that the number of images in both zones is the same
def count_images(bucket_name, prefix):
    count = 0

    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]

                # skip folders / hidden/system files
                if not key.endswith("/") and not key.startswith(("_", ".")):
                    count += 1

    return count

landing_count = count_images("landing-zone", "persistent-landing/unstructured/image/")
trusted_count = count_images("trusted-zone", "persistent-landing/unstructured/image/")

print("Landing zone images:", landing_count)
print("Trusted zone images:", trusted_count)

Landing zone images: 32303
Trusted zone images: 32303


In [6]:
# Replicate the file catalogue for later use
replicate_bucket(src_bucket = "landing-zone", src_prefix="persistent-landing/structured/file_catalog", dest_bucket = "trusted-zone")

**Applying Transformations**

In [7]:
# Setup SparkSession
spark = SparkSession.builder \
    .appName("trusted_zone-unstructured") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.maximum", "50") \
    .config("spark.hadoop.fs.s3a.threads.max", "50") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .getOrCreate()

In [8]:
# Due to version mismatches, some variables have values like 60s that need to be normalized
conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in conf.iterator():
    k = item.getKey()
    v = item.getValue()

    # normalize only simple cases like "60s" → "60"
    if isinstance(v, str):
        if v.endswith("s") or v.endswith("h"):
            digits = "".join(c for c in v if c.isdigit())
            conf.set(k, digits)

In [19]:
# Get image list
paginator = s3.get_paginator("list_objects_v2")

files = []

for page in paginator.paginate(
    Bucket="trusted-zone",
    Prefix="persistent-landing/unstructured/image/"
):
    for obj in page.get("Contents", []):
        if not obj["Key"].endswith("/"):
            files.append(f"s3a://trusted-zone/{obj['Key']}")

In [21]:
# Function that transforms an image given the path
def transform_and_upload(path):
    try:
        # -----------------------------
        # Read directly from MinIO via boto3
        # -----------------------------
        s3 = boto3.client(
            "s3",
            endpoint_url=endpoint,
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key
        )

        bucket = "trusted-zone"
        key = path.replace("s3a://trusted-zone/", "")

        # -----------------------------
        # Load image
        # -----------------------------
        obj = s3.get_object(Bucket=bucket, Key=key)
        data = obj["Body"].read()

        img = Image.open(io.BytesIO(data))

        # corrupted image handling
        try:
            img.verify()
        except Exception:
            return None

        img = Image.open(io.BytesIO(data)).convert("RGB")

        # -----------------------------
        # Normalize size (no rescaling logic change)
        # -----------------------------
        img = img.resize((512, 512))

        # -----------------------------
        # Convert to PNG
        # -----------------------------
        buffer = io.BytesIO()
        img.save(buffer, format="PNG")
        buffer.seek(0)

        new_key = key.rsplit(".", 1)[0] + ".png"

        # -----------------------------
        # Write new file
        # -----------------------------
        s3.put_object(
            Bucket=bucket,
            Key=new_key,
            Body=buffer.getvalue(),
            ContentType="image/png"
        )

        # -----------------------------
        # DELETE OLD FILE (important part)
        # -----------------------------
        s3.delete_object(
            Bucket=bucket,
            Key=key
        )

        return new_key

    except Exception:
        return None

In [22]:
# Spark parallel processing RDD over the list of images
rdd = spark.sparkContext.parallelize(files)
result = rdd.map(transform_and_upload).filter(lambda x: x is not None)

In [23]:
# Action to trigger execution of transformations
result_list = result.collect()

In [13]:
# At this moment, all buckets should have the same number of files. Let's check that

# Count valid files in each bucket
for bucket in s3.list_buckets()["Buckets"]:
    
    bucket_name = bucket["Name"]
    count = 0

    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix="persistent-landing/unstructured/image/"):
        
        if "Contents" in page:
            for obj in page["Contents"]:
                
                key = obj["Key"]
                
                # Skip folder placeholders (end with "/") and metadata/system files
                if not key.endswith("/") and not key.startswith(("_", ".")):
                    count += 1

    print(f"Bucket: {bucket_name}, Files: {count}")

Bucket: landing-zone, Files: 32303
Bucket: trusted-zone, Files: 32303
